[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week01.ipynb)

# Week 1: Deep Learning Overview & ML-to-Network Framing
**Introduction to Deep Learning (HIT)** &middot; Part I: Foundations

What deep learning is; framing a task as tensor inputs, model outputs, and a loss function.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Set up the PyTorch toolchain and confirm it runs.
- Frame any ML task as tensors in, a model, and a loss out.
- Run a first end-to-end training loop and read the loss curve.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Set up PyTorch live and confirm the device (GPU or CPU).

In [ ]:
# Confirm the device with a small tensor
x = torch.randn(3, device=device)
print("device:", x.device, "| tensor:", x)

## Demonstration 2
Walk through a minimal training loop on a toy dataset, run it, and read the loss curve.

In [ ]:
# Minimal training loop on a toy linear dataset y = 2x + 1 + noise
torch.manual_seed(0)
X = torch.randn(200, 1)
y = 2 * X + 1 + 0.1 * torch.randn(200, 1)

model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(50):
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
    losses.append(loss.item())

print("learned weight, bias:", model.weight.item(), model.bias.item())
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.title("Training loss"); plt.show()

## Demonstration 3
Vary the learning rate live to show divergence versus convergence.

In [ ]:
# Same problem, three learning rates: convergence vs divergence
def train(lr, steps=50):
    torch.manual_seed(0)
    m = nn.Linear(1, 1); o = torch.optim.SGD(m.parameters(), lr=lr); f = nn.MSELoss()
    hist = []
    for _ in range(steps):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); hist.append(l.item())
    return hist

for lr in [0.01, 0.1, 1.5]:
    plt.plot(train(lr), label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log)"); plt.legend(); plt.show()

## Demonstration 4
Frame a classification and a regression example as tensors-in, loss-out, in code.

In [ ]:
# Framing: classification (logits + cross-entropy) vs regression (value + MSE)
xb = torch.randn(8, 4)                       # 8 samples, 4 features

clf = nn.Linear(4, 3)                        # -> 3 class logits
ce = nn.CrossEntropyLoss()(clf(xb), torch.randint(0, 3, (8,)))
print("classification: logits", tuple(clf(xb).shape), "| CE loss", round(ce.item(), 3))

reg = nn.Linear(4, 1)                        # -> 1 continuous value
mse = nn.MSELoss()(reg(xb), torch.randn(8, 1))
print("regression: output", tuple(reg(xb).shape), "| MSE loss", round(mse.item(), 3))

---
Student materials for this week: the lab handout (`labs/week01.html`) and the curated references (`references/week01.html`) in the course site.